##### Copyright 2023 Google LLC

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## Setup

In [ ]:
!pip install -U -q "google-generativeai>=0.8.2"

In [12]:
# import necessary modules.

import google.generativeai as genai

import base64
import json

try:
    # Mount google drive
    from google.colab import drive

    drive.mount("/gdrive")

    # The SDK will automatically read it from the GOOGLE_API_KEY environment variable.
    # In Colab get the key from Colab-secrets ("🔑" in the left panel).
    import os
    from google.colab import userdata

    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass

# Parse the arguments

model = "gemini-1.5-flash"  # @param {isTemplate: true}
contents_b64 = b'W3sicGFydHMiOlt7InRleHQiOiJTdW1tYXJpemUgOiDnubDjgorov5TjgZfmj5DmoYjjgZXjgozjgovljbHmqZ/nrqHnkIbpg6jploDjga7pm4bntITmp4vmg7NcbiAgICAgIOOAgDIwMjTlubQxMOaciOOBq+WwseS7u+OBl+OBn+efs+egtOiMgue3j+eQhuOBr+OAgeWGhemWo+OBruebrueOieaUv+etluOBruS4gOOBpOOBqOOBl+OBpuOAjOmYsueBveW6geOAjeOBruioree9ruOCkuaOsuOBkuOBpuOBhOOCi+OAguWQjOW5tDEy5pyI44Gr5Yid6ZaL5YKs44GV44KM44Gf6Ziy54G956uL5Zu95o6o6YCy6Zaj5YOa5Lya6K2w44Gn44Gv44CBMjAyNuW5tOW6puS4reOBruWQjOW6geioree9ruOCkuebruaMh+OBmeaWuemHneOBjOaOsuOBkuOCieOCjOOBn+OBjOOAgeWFt+S9k+eahOOBquWItuW6puioreioiOOBr+ekuuOBleOCjOOBpuOBhOOBquOBhOOAglxuICAgICAg44CA5pel5pys44Gr44GK44GE44Gm44CB5ZCE55yB44Gr5YiG5pWj44GX44Gm44GE44KL54G95a6z5a++5b+c5qip6ZmQ44KS6ZuG57SE44GX44KI44GG44Go44Gu6ICD44GI44Gv5aSn6KaP5qih54G95a6z44Gu44Gf44Gz44Gr5o+Q5Ye644GV44KM44Gm44GN44Gf44CCMjAxMeW5tOOBruadseaXpeacrOWkp+mch+eBveW+jOOBq+OBr+OAgeexs+WbveOBrumAo+mCpue3iuaApeS6i+aFi+euoeeQhuW6ge+8iEZFTUHvvInjgpLlj4LogIPjgavjgIHngb3lrrPlr77lv5zjgavnibnljJbjgZfjgZ/nnIHluoHjgpLoqK3nq4vjgZnjgovjgbnjgY3jgajjga7mhI/opovjgYzlm73kvJrjgafnubDjgorov5TjgZfmj5DotbfjgZXjgozjgZ/jgIIyMDIw772eMjAyMuW5tOOBrkNPVklELTE544Gu5LiW55WM55qE44G+44KT5bu25b6M44Gr44KC44CB5Yy755mC44CB5pWR5oCl5a++5b+c44CB5aSW5Ye644CB5Za25qWt6Ieq57Kb44Gq44Gp44Gu56S+5Lya5a++5b+c44KS6L+F6YCf44Gr5bGV6ZaL44Gn44GN44KL57WE57mU5L2c44KK44GM6K2w6KuW44GV44KM44CBMjAyM+W5tDnmnIjjgIHlhoXplqPlrpjmiL/jgavmlrDjgZ/jgarntYTnuZTjgYzoqK3nq4vjgZXjgozjgZ/jgILph43opoHjgqTjg7Pjg5Xjg6njgbjjga7lpKfopo/mqKHjgrXjgqTjg5Djg7zmlLvmkoPjgarjganmlrDjgZ/jgarkuovmhYvnmbrnlJ/jga7lj6/og73mgKfjgoLlj5bjgorjgZbjgZ/jgZXjgozjgIHljbHmqZ/nrqHnkIbog73lipvjgpLlkJHkuIrjgZXjgZvjgovjgZPjgajjga/lm73lrrboqrLpoYzjga7kuIDjgaTjgavjgarjgaPjgabjgYTjgovjgIJcbiAgICAgIOOAgOWNseapn+euoeeQhuOBruWwgumWgOWutuOBp+ani+aIkOOBmeOCi+esueW3neW5s+WSjOiyoeWbo+WuieWFqOS/nemanOeglOeptuOCsOODq+ODvOODl+OBruODl+ODreOCuOOCp+OCr+ODiOOAjOe3iuaApeS6i+aFi+azleWItueglOeptuS8muOAjeOBp+OBr+OAgeOBk+OBhuOBl+OBn+S6i+aDheOCkui4j+OBvuOBiOOAgeecgeW6geS9k+WItuOBruOBguOCiuaWueOCkuWQq+OCgeOAgee3iuaApeS6i+aFi+WvvuWHpuOBruWun+WKueaAp+WQkeS4iuOBruaWueetluOCkuitsOirluOBl+OBpuOBjeOBn+OAguOBneOBruaIkOaenOOCkueUn+OBi+OBl+OAgeacrOeov+OBp+OBr+OAgeaXpeacrOOBq+OBiuOBkeOCi+ePvuihjOOBrueBveWus+WvvuW/nOS9k+WItuOCkuamguims+OBl+OAgea1t+WkluS6i+S+i+OBqOOBl+OBpkZFTUHjgpLliIbmnpDjgZfjgZ/lvozjgIHpmLLngb3luoHoqK3nva7mp4vmg7PjgafogIPmha7jgZnjgbnjgY3jgZPjgajjgpLmjJnjgZLjgovjgIJcblxuICAgICAg5pel5pys44Gr44GK44GR44KL6Ziy54G944CB54G95a6z5a++5b+c5L2T5Yi244Gu5qaC6KaBXG4gICAgICDjgIDml6XmnKzjga7pmLLngb3jgIHngb3lrrPlr77lv5zjga/jgIHlhoXplqPlupzpmLLngb3pg6jploDjgpLkuK3lv4PjgavjgIHngb3lrrPjga7nqK7liKXjgavjgojjgaPjgabkuLvjgarlvbnlibLjgpLmi4XjgYbntYTnuZTjgYznlbDjgarjgovjgIJcbiAgICAgIOOAgOWGhemWo+W6nOmYsueBvemDqOmWgOOBr+e0hDE1MOS6uuOBruS9k+WItuOBp+mYsueBveioiOeUu+OBruetluWumuOChOiok+e3tOOAgemBv+mbo+eUn+a0u+aUr+aPtOOBruS9k+WItuOBpeOBj+OCiuOBq+W9k+OBn+OBo+OBpuOBhOOCi+OAguS4gOWumuimj+aooeOBrueBveWus+OBjOi1t+OBk+OCjOOBsOOAgeWvvuetluacrOmDqOOBruioree9ruOBq+W+k+S6i+OBl+OAgeWGhemWo+WumOaIv+OBruWGhemWo+WNseapn+euoeeQhuebo+OCkuS4reW/g+OBqOOBmeOCi+WvvuW/nOOCkuijnOWujOOBmeOCi+OAguOBguOCj+OBm+OBpuOAgemWoumAo+ecgeW6geOAgeiiq+eBvemDvemBk+W6nOecjOOAgeiHquayu+S9k+OBqOOBruiqv+aVtOOCkuihjOOBhuOAguWGhemWo+W6nOOBr+WGhemWo+OBq+ebtOWxnuOBl+OAgeeJueWumuOBruWIhumHjuOChOalreeVjOOCkuaLheW9k+OBmeOCi+S7luecgeW6geOBi+OCieeLrOeri+OBl+OBpuOBhOOCi+OBn+OCgeOAgeiqv+aVtOOCkuihjOOBhOOChOOBmeOBhOeri+WgtOOBq+OBguOCi+OAglxuICAgICAg44CA5LiK6KiY44Gv6Ieq54S254G95a6z44KS5oOz5a6a44GX44Gf5L2T5Yi244Gn44GC44KK44CB6auY5bqm44Gq5bCC6ZaA55+l6K2Y44GM5b+F6KaB44Gq54G95a6z44G444Gu5a++5b+c44Gv55Ww44Gq44Gj44Gm44GE44KL44CC5Y6f5a2Q5Yqb54G95a6z44G444Gu5a++5b+c44Gv44CB5p2x5Lqs6Zu75Yqb56aP5bO256ys5LiA5Y6f55m65LqL5pWF44KS5Y+X44GR44Gm5paw44Gf44Gr5pW05YKZ44GV44KM44Gf44CC5LqL5pWF55m655Sf5pmC44Gv5Y6f5a2Q5Yqb44Gu5bCC6ZaA55+l6K2Y44KS5pyJ44GZ44KL5Y6f5a2Q5Yqb6KaP5Yi25aeU5ZOh5Lya44CB5Y6f5a2Q5Yqb6KaP5Yi25bqB44Gn5qeL5oiQ44GZ44KL57eK5oCl5pmC5a++5b+c44K744Oz44K/44O844OB44O844Og44GM5b2T6Kmy5pa96Kit77yI44Kq44Oz44K144Kk44OI77yJ44Gn44Gu5a++5b+c44KS5pSv5o+044GZ44KL44CC5pS/5bqc5a++562W5pys6YOo44Gu5LqL5YuZ5bGA44Gv5YaF6Zaj5bqc44Gr572u44GL44KM44CB44K744Oz44K/44O844OB44O844Og44Go6YCj5pC644GX44Gk44Gk44CB5pa96Kit5aSW77yI44Kq44OV44K144Kk44OI77yJ44Gn44Gu5a++5b+c44KS5ouF44GG44CC44Kq44OV44K144Kk44OI5a++5b+c44Gv44CB5L2P5rCR6YG/6Zuj44Gu44Gf44KB44CB6Ly46YCB5omL5q6144KE6YG/6Zuj5YWI44Gu56K65L+d44Gn55yB5bqB44CB6Ieq5rK75L2T44CB6YGL6Ly45Lya56S+44Gq44Gp5aSa44GP44Gu5qmf6Zai44Go44Gu6Kq/5pW044GM5b+F6KaB44Gr44Gq44KL77yI5ZuzMe+8ieOAguOBneOBruOBn+OCgeOAgeWGhemWo+W6nOaUv+etlue1seaLrOWumOOBjOe0hDUw5ZCN44Gu6IG35ZOh44Gu44K144Od44O844OI44KS5b6X44Gm5a++5b+c44GZ44KL44CCXG4gICAgICDjgIDjgZfjgYvjgZfjgarjgYzjgonjgIHlhoXplqPlupzjgYzjgZnjgbnjgabjga7ngb3lrrPjgavjgYrjgYTjgaboqr/mlbTmqZ/og73jgpLmnpzjgZ/jgZnjgo/jgZHjgafjga/jgarjgYTjgILjg5Hjg7Pjg4fjg5/jg4Pjgq/jgavjgaTjgYTjgabjga/jgIFDT1ZJRC0xOeOBruOBvuOCk+W7tuOCkuWPl+OBkeOBpuOAgeWGhemWo+WumOaIv+OBq+WGhemWo+aEn+afk+eXh+WNseapn+euoeeQhue1seaLrOW6geOCkuaWsOioreOBl+aEn+afk+eXh+OBq+mWouOBmeOCi+WvvuW/nOOCkumbhue0hOOBl+OBn++8iOWbszLvvInjgILmlL/lupzlr77nrZbmnKzpg6jjgYzoqK3nva7jgZXjgozjgZ/loLTlkIjjgIHlkIzluoHjgYzlkITnnIHluoHjgoToh6rmsrvkvZPjgIHkv53lgaXmiYDjgajjga7oqr/mlbTjgpLkuIDlhYPnmoTjgavmi4XjgYbjgILjgrXjgqTjg5Djg7zmlLvmkoPjgbjjga7lr77lh6bjga/jgIEyMDE15bm044Gr5YaF6Zaj5a6Y5oi/44Gr5paw6Kit44GV44KM44Gf5YaF6Zaj44K144Kk44OQ44O844K744Kt44Ol44Oq44OG44Kj44K744Oz44K/44O877yITklTQ++8ieOBjOS4reW/g+OBqOOBquOCi+OAglxuICAgICAg5ZuzIDLvvJrlhoXplqPmhJ/mn5Pnl4fljbHmqZ/nrqHnkIbntbHmi6zluoHjgpLkuK3lv4PjgajjgZfjgZ/lj7jku6TloZTmqZ/og73jga7lvLfljJZcbiAgICAgIOOAgOOBk+OBruOCiOOBhuOBq+ePvuihjOOBruaXpeacrOOBruS7lee1hOOBv+OBr+OAgeWNseapn+WvvuW/nOOAgeiqv+aVtOapn+iDveOCkuaenOOBn+OBmeapn+mWouOBjOeBveWus+OBrueoruWIpeOBq+OCiOOCiuOAgee0sOOBi+OBj+WIhuOBi+OCjOOBpuOBhOOCi+OAguitpuWvn++8iOmDvemBk+W6nOecjO+8ieOAgea2iOmYsu+8iOW4gueUuuadke+8ieOAgeiHquihm+maiu+8iOWbve+8ieOBruOBu+OBi+OAgeeBveWus+a0vumBo+WMu+eZguODgeODvOODoO+8iERNQVTjgIHljprnlJ/lirTlg43nnIHvvInjgIHpgZPot6/jgoTloKTpmLLjga7lvqnml6fjgpLmlK/mj7TjgZnjgovnt4rmgKXngb3lrrPlr77nrZbmtL7pgaPpmorvvIhURUMtRk9SQ0XjgIHlm73lnJ/kuqTpgJrnnIHvvInjgarjganjgIHooqvngb3lnLDjgaflrp/lg43pg6jpmorjgajjgarjgovkurrlk6HjgoLmiYDnrqHjgYzliIbjgYvjgozjgabjgYTjgovjgILjgb7jgZ/jgIHkurrlk6Hnt4/mlbDjgoLjgIHljbHmqZ/nrqHnkIbjga7lsILploDnn6XorZjjgpLmjIHjgaPjgZ/ogbflk6HjgoLkuI3otrPjgZfjgabjgYTjgovjgILngb3lrrPjgYznmbrnlJ/jgZnjgozjgbDjgIHlhoXplqPlupzpmLLngb3pg6jploDjga/jgbvjgbzlhajlk6Hnt4/lh7rjgaflr77lv5zjgavlsILlv7XjgZnjgovjgZ/jgoHjgIHpmLLngb3ln7rmnKzoqIjnlLvjga7nrZblrprjgarjganlubPmmYLjga7mpa3li5njga/kuK3mlq3jgpLkvZnlhIDjgarjgY/jgZXjgozjgovjgIIyMDI05bm05LiK5Y2K5pyf44Gr44Go44KK44G+44Go44KB44KL44Gv44Ga44Gg44Gj44Gf5Y2X5rW344OI44Op44OV5Zyw6ZyH44Gu5Z+65pys6KiI55S76KaL55u044GX44Gv44CB5ZCM5bm077yR5pyI44Gr55m655Sf44GX44Gf6IO955m75Y2K5bO25Zyw6ZyH44Gu5a++5b+c44Gr6L+944KP44KM44CB5aSn5bmF44Gr6YGF44KM44Gm44GE44KL44CCXG4gICAgICDmtbflpJbjga7ngb3lrrPlr77lv5zkvZPliLbvvJrnsbPlm71GRU1B44Gu5L2T5Yi25YiG5p6QXG4gICAgICDjgIDnsbPlm73jga5GRU1B44Gv57eK5oCl5pSv5o+05qWt5YuZ77yIRW1lcmdlbmN5IFN1cHBvcnQgRnVuY3Rpb25z77yaRVNGc++8ieOCkjE144Gr5YiG6aGe44GX44CB44Gd44Gu5a6f6KGM44Gr44Gk44GE44Gm44CB5Li76KaB5ouF5b2T55yB5bqB77yIUO+8ieOAgeOCteODneODvOODiOecgeW6ge+8iFPvvInjgIHoqr/mlbTmqZ/plqLvvIhD77yJ44Gr5pW055CG44GX44Gm44GE44KL44Gu44GM54m55b6044Gn44GC44KL44CC5b255Ymy5YiG5ouF44KS5piO56K644Gr44GX44CB6L+F6YCf44Gq5Y2x5qmf5a++5b+c44KS5Zuz44KN44GG44Go44GX44Gm44GE44KL77yI6KGoMe+8ieOAgjE5NznlubQ05pyI44CB5raI6Ziy5bqB44CB6YCj6YKm54G95a6z5o+05Yqp5bqB44Gq44GpNuOBpOOBruW6geOAgeWxgOOCkue1seWQiOOBl+OAgeeLrOeri+OBl+OBn+ecgeW6geOBqOOBl+OBpuioreeri+OBleOCjOOBn+OAglxuICAgICAg44CARVNGc+OBruOBhuOBoeOAgUZFTUHjga/oh6rnhLbngb3lrrPjga7liJ3li5Xlr77lv5zjgafnibnjgavph43opoHjgajjgZXjgozjgovpgJrkv6HjgIHmg4XloLHjg7voqIjnlLvjgIHooqvngb3ogIXlr77lv5zjgIHjg63jgrjjgrnjg4bjgqPjgq/jgrnjgIHmjZzntKLjg7vmlZHliqnjgIHlr77lpJbluoPloLHjga7vvJbmpa3li5njgavnibnljJbjgZnjgovjgILmrovjgoo55qWt5YuZ44Gv5ZCE55yB5bqB44Go44Gu6Kq/5pW044KS57WM44Gm5a6f5pa944GV44KM44KL44CC44G+44Gf44CB5Y6f5a2Q5Yqb44CB44K144Kk44OQ44O85LqL5qGI44CB44OR44Oz44OH44Of44OD44Kv44Gq44Gp44Gv5bCC6ZaA55+l6K2Y44KS5pyJ44GZ44KL55yB5bqB44GM5a++5b+c44KS5Li75bCO44GX44CBRkVNQeOBr+S9j+awkemBv+mbo+OAgeODreOCuOOCueODhuOCo+OCr+OCueOCkuaLheW9k+OBmeOCi+OAglxuICAgICAg44CARkVNQeOBr+WNseapn+euoeeQhuOBruWwgumWgOefpeitmOOCkuaMgeOBoeOAgemAmuS/oeOChOW7uuevieWcn+acqOOAgeaVkeaApeWvvuW/nOOBq+W/heimgeOBquizh+agvOOChOWFjeioseOCkuacieOBmeOCi+e0hDcsNTAw5ZCN44Gu6KaB5ZOh44KS5oqx44GI44KL44CC5YWo57Gz44GuMTDjgYvmiYDjgavoqK3nva7jgZfjgZ/lnLDmlrnmi6DngrnjgYvjgonjgIHjgZPjgYbjgZfjgZ/lrp/li5Xpg6jpmorjgYzooqvngb3lvozjgZnjgZDjgavmtL7pgaPjgZXjgozjgovjgILlubPmmYLjga/jgIHlkITlt57jgIHlkIToh6rmsrvkvZPjgYznrZblrprjgZnjgovpmLLngb3oqIjnlLvnrZblrprjgbjjga7liqnoqIDjgIHpgKPpgqbmlL/lupzvvI3lt57vvI3oh6rmsrvkvZPjgpLkuqTjgYjjgZ/oqJPnt7Tjga7jgbvjgYvjgIHkvIHmpa3jgIFOUE/jgajjga7ngb3lrrPljZTlipvljZTlrprjga7nt6DntZDjgpLooYzjgYbjgIJcbiAgICAgIOOAgOOBl+OBi+OBl+OBquOBjOOCieOAgUZFTUHjga7kvZPliLbjgYzkuI3lpInjgaDjgaPjgZ/jgo/jgZHjgafjga/jgarjgYTjgIIyMDAx5bm0OeaciDEx5pel44Gr55m655Sf44GX44Gf5ZCM5pmC5aSa55m644OG44Ot5LqL5Lu244Gr5aSa5aSn44Gq5b2x6Z+/44KS5Y+X44GR44Gf6Ium44GE57WM6aiT44GM44GC44Gj44Gf44GT44Go44GM55+l44KJ44KM44Gm44GE44KL44CCXG4gICAgICDjgIDljbHmqZ/nrqHnkIbjga7lsILploDlrrbjgafjgYLjgovkvIrol6TmvaTjgavjgojjgovjgajjgIHlkIzkuovku7bjgafjga/jgIHkuK3lpK7mg4XloLHlsYDvvIhDSUHvvInjgYzkuovliY3jgavlhaXmiYvjgZfjgabjgYTjgZ/ph43opoHmg4XloLHjgpLmlL/lupzlhoXjgaflhbHmnInjgafjgY3jgZrjgIHooqvlrrPlr77lv5zjgoLkuI3ljYHliIbjgaDjgaPjgZ/jgILjgZ3jga7jgZ/jgoHjgIHjg5bjg4Pjgrfjg6XlpKfntbHpoJjvvIjlvZPmmYLvvInjga/ntYTnuZTmlLnnt6jjgavkuZfjgorlh7rjgZfjgIEyMDAz5bm077yR5pyI44CB5Zu95Zyf5a6J5YWo5L+d6Zqc5bGA44CB5rK/5bK46K2m5YKZ6ZqK44CBRkVNQeOBquOBqTIy44Gu5pS/5bqc5qmf6Zai44KS57Wx5ZCI44GX44CB6Zaj5YOa44GM6ZW35a6Y44KS5YuZ44KB44KL5pS/5bqc5qmf6Zai44Go44GX44Gm44Gv44CB5ZCM5Zu9MTXnlarnm67jga7nnIHjgajjgZfjgablm73lnJ/lronlhajkv53pmpznnIHvvIhESFPvvInjgYzlibXoqK3jgZXjgozjgZ/jgIJGRU1B44Gv54us56uL44KS5aSx44GE44CB5Lq65LqL44CB5LqI566X5qip44GM5Yi257SE44GV44KM44Gf44CCXG4gICAgICDjgIDjg4bjg63lr77nrZbkuK3lv4Pjga7ntYTnuZTkvZPliLbjga/ku5bjga7ngb3lrrPjgbjjga7lr77lv5zlipvjgpLlvLHkvZPljJbjgZXjgZvjgZ/jgIIyMDA15bm077yY5pyI44CB44OP44Oq44Kx44O844Oz44O744Kr44OI44Oq44O844OK44GM44Or44Kk44K444Ki44OK5bee44Gq44Gp44Gr55Sa5aSn44Gq6KKr5a6z44KS44KC44Gf44KJ44GX44Gf44CC44GT44Gu6Zqb44CB6Ieq5rK75L2T77yN5bee77yN6YCj6YKm5pS/5bqc6ZaT44Gu6YCj5L+C44Of44K544Gr44KI44KL5pSv5o+054mp6LOH44CB5Lq65ZOh5rS+6YGj44Gu6YGF44KM44GM55u45qyh44GO44CB5Y6z44GX44GP5om55Yik44GV44KM44Gf44CCREhT55m66Laz44Gr5Ly044GG44OG44Ot5a++562W5YGP6YeN5Lq65LqL44Gr44KI44KK44CB6Ieq54S254G95a6z5a++5b+c44G444Gu5bCC6ZaA55+l6K2Y44KS5oyB44Gj44Gf6IG35ZOh44GMRkVNQeOBruS4u+imgeODneOCueODiOOBi+OCieWkluOBleOCjOOBpuOBhOOBn+OBk+OBqOOBjOWOn+WboOOBqOaMh+aRmOOBleOCjOOBn+OAguOBk+OBruS6i+aFi+OCkuWPl+OBkeOAgTIwMDblubTjgIHjg53jgrnjg4jjg7vjgqvjg4jjg6rjg7zjg4rnt4rmgKXkuovmhYvnrqHnkIbmlLnpnanms5XjgYzliLblrprjgZXjgozjgIFGRU1B44Gv5Lq65LqL44CB5LqI566X5qip44Gq44Gp54us56uL57WE57mU44Go44GX44Gm44Gu5Zyw5L2N44KS5Y+W44KK5oi744GX44GfXG4ifV19XQ=='
generation_config_b64 = "e30="  # @param {isTemplate: true}
safety_settings_b64 = "e30="  # @param {isTemplate: true}

contents = json.loads(base64.b64decode(contents_b64))

generation_config = json.loads(base64.b64decode(generation_config_b64))
safety_settings = json.loads(base64.b64decode(safety_settings_b64))

stream = False

print(json.dumps(contents, indent=4))

Drive already mounted at /gdrive; to attempt to forcibly remount, call drive.mount("/gdrive", force_remount=True).
[
    {
        "parts": [
            {
                "text": "Summarize : \u7e70\u308a\u8fd4\u3057\u63d0\u6848\u3055\u308c\u308b\u5371\u6a5f\u7ba1\u7406\u90e8\u9580\u306e\u96c6\u7d04\u69cb\u60f3\n      \u30002024\u5e7410\u6708\u306b\u5c31\u4efb\u3057\u305f\u77f3\u7834\u8302\u7dcf\u7406\u306f\u3001\u5185\u95a3\u306e\u76ee\u7389\u653f\u7b56\u306e\u4e00\u3064\u3068\u3057\u3066\u300c\u9632\u707d\u5e81\u300d\u306e\u8a2d\u7f6e\u3092\u63b2\u3052\u3066\u3044\u308b\u3002\u540c\u5e7412\u6708\u306b\u521d\u958b\u50ac\u3055\u308c\u305f\u9632\u707d\u7acb\u56fd\u63a8\u9032\u95a3\u50da\u4f1a\u8b70\u3067\u306f\u30012026\u5e74\u5ea6\u4e2d\u306e\u540c\u5e81\u8a2d\u7f6e\u3092\u76ee\u6307\u3059\u65b9\u91dd\u304c\u63b2\u3052\u3089\u308c\u305f\u304c\u3001\u5177\u4f53\u7684\u306a\u5236\u5ea6\u8a2d\u8a08\u306f\u793a\u3055\u308c\u3066\u3044\u306a\u3044\u3002\n      \u3000\u65e5\u672c\u306b\u304

## Call `generate_content`

In [13]:
from IPython.display import display
from IPython.display import Markdown
from google.colab import userdata
userdata.get('GOOGLE_API_KEY')

# Call the model and print the response.
gemini = genai.GenerativeModel(model_name=model)
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

response = gemini.generate_content(
    contents,
    generation_config=generation_config,
    safety_settings=safety_settings,
    stream=stream,
)

display(Markdown(response.text))

This paper examines the repeatedly proposed consolidation of Japan's disaster management agencies, focusing on the planned establishment of a new "Disaster Prevention Agency" (防災庁).  While the idea of consolidating disaster response powers, currently scattered across various ministries, has been raised after every major disaster (including the 2011 Tohoku earthquake and the COVID-19 pandemic),  concrete plans remain elusive.  The paper compares Japan's current fragmented system – where responsibility varies by disaster type (e.g., natural disasters handled by the Cabinet Office, pandemics by a newly established agency within the Cabinet Secretariat, nuclear accidents by the Nuclear Regulation Authority) – with the US Federal Emergency Management Agency (FEMA).

The analysis highlights FEMA's structure, which categorizes emergency support functions and designates lead and supporting agencies.  However, the paper notes FEMA's evolution, particularly its integration into the Department of Homeland Security after 9/11, resulting in a temporary weakening of its natural disaster response capabilities due to a shift towards counter-terrorism.  The subsequent Hurricane Katrina response highlighted the pitfalls of this reorganization.  The paper concludes by suggesting considerations for the proposed Japanese Disaster Prevention Agency, drawing lessons from both Japan's past experiences and FEMA's successes and failures, emphasizing the need for sufficient staffing, specialized expertise, and clear inter-agency coordination to ensure effective disaster response.


<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://ai.google.dev/gemini-api/docs"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />Docs on ai.google.dev</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google-gemini/cookbook/blob/main/quickstarts"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />More notebooks in the Cookbook</a>
  </td>
</table>